# Summary: SoC estimator comparison

Collects the saved tables from `controlled_evaluation.ipynb` (run for both batteries) and `operational_evaluation.ipynb` (run for every run in `soc_eval.OPERATIONAL_RUNS`). It builds the cross-battery figure and the report's summary tables, and it runs no filters itself.

How to read the two halves:
- **Controlled runs** are characterization: in-sample, against a characterization SoC that is not ground truth.
- **Operational runs** compare the methods with each other, because they have no reference SoC.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'final_report' / 'soc_eval.py').exists()) / 'final_report'))
import soc_eval as se

se.apply_style()
pd.set_option('display.precision', 3)
pd.set_option('display.width', 200)
T = se.TABLE_DIR
BATTERIES = [b for b in se.BATTERIES if (T / f'{b}_controlled_scores.csv').exists()]
RUNS = [r for r in se.OPERATIONAL_RUNS if (T / f"op_{r.replace(' ', '_')}_summary.csv").exists()]
print('Controlled results found for:', BATTERIES)
print('Operational results found for:', RUNS)

## Figure: characterization of both batteries
Each method's difference from the characterization SoC at the end of every rest, plotted against SoC.
- OCV mapping is on the top row, on its own scale.
- On LiFePO4 the flat OCV curve makes OCV mapping unusable between about 18% and 84% SoC.

In [ ]:
divs = {b: pd.read_csv(T / f'{b}_controlled_divergence.csv') for b in BATTERIES}
fig = se.plot_characterization(divs)
se.save(fig, 'summary_characterization')
plt.show()

## Table 1: characterization quality (controlled runs, tuned settings)

In [ ]:
scores = pd.concat({b: pd.read_csv(T / f'{b}_controlled_scores.csv', index_col=0) for b in BATTERIES}, names=['battery'])
se.save_table(scores, 'summary_characterization')
display(scores)

## Table 2: input current, raw vs offset-corrected (controlled runs)

In [ ]:
inputs = pd.concat({b: pd.read_csv(T / f'{b}_controlled_input_current.csv', index_col=0) for b in BATTERIES}, names=['battery'])
se.save_table(inputs, 'summary_input_current')
display(inputs)

## Table 3: before and after tuning (controlled runs)
The original settings are the server as it was:
- process noise from `noise.json`;
- P0 = 1e-8;
- R = σ_v² per sample;
- raw current;
- the repo KCL corrector for EKF + KCL.

In [ ]:
files = {b: T / f'{b}_controlled_before_tuning.csv' for b in BATTERIES}
before = pd.concat({b: pd.read_csv(f, index_col=[0, 1]) for b, f in files.items() if f.exists()}, names=['battery'])
se.save_table(before, 'summary_before_tuning')
display(before)

## Table 4: operational runs
These are disagreement between methods, stability and recovery, not accuracy. One column per run.

In [ ]:
operational = pd.concat([pd.read_csv(T / f"op_{r.replace(' ', '_')}_summary.csv", index_col=0) for r in RUNS], axis=1)
operational = operational.map(lambda v: round(float(v), 3) if str(v).replace('.', '', 1).replace('-', '', 1).replace('e', '', 1).isdigit() else v)
se.save_table(operational, 'summary_operational')
display(operational)

## Table 5: KCL evidence, the 29 Jun run as the server processed it

In [ ]:
f = T / 'op_LiNMC_Battery_Discharge_Test_latch_evidence.csv'
if f.exists():
    display(pd.read_csv(f, index_col=0))
else:
    print('Run operational_evaluation.ipynb with RUN = "LiNMC General Test" first.')